# 0. WATERec options and experiment findings

Start with `README.md`, `docs/capabilities.md` and `docs/experiments.md`. R and RS are equal-size pretrained variants (26.12M parameters), not a Nano/Tiny size ladder. Price SPACE adds 768 parameters. PARSeq Fast Tiny is maintained in its separate repository.

Historical token-selection checkpoints reached159/206 and later161/206 literal exact. The restart2 architecture has17,913,953 parameters, a2,048-patch input budget,512 selected patches after six blocks and two AR layers. These are different checkpoints/training histories, not one lossless model update. Standard R/RS options are unchanged; notebook9 exposes the separate experimental workflow. Reference weights are not bundled.

Current training/evaluation policy: **literal-string exact only**, including spaces, symbols and case; earliest tied checkpoint wins. TXT permits one framing LF/CRLF only. Old training-selection state is incompatible with resume; historical reference cells retain their original scope. See [literal text policy](../docs/literal_text_policy.md).

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
reference = json.loads((ROOT / "results/reference_metrics.json").read_text())
print(json.dumps(reference["model_options"], indent=2))


Choose `WATEREC_CONFIG=configs/dummy_price.yaml` for fictional mixed-size prices. Real-data starting profiles are `waterec_alpr.yaml`, `waterec_r_alpr.yaml`, `waterec_alpr_1024.yaml`, `waterec_price.yaml`, and `waterec_price_512.yaml`. Changing budget requires separate quality checks and matching exports. Default native labels are at most 24 characters with 25 decoder steps including EOS.

## Historical compact research and the experimental workflow

See [the compact research update](../docs/compact_research.md) and [aggregate evidence](../results/compact_research.json). The image-only token selector uses 2048 input patches and 512 late encoder/decoder memory tokens, 17,913,953 parameters, and image/text-only training. For the original terminal, the earlier runtime measured 1.422 ms RGB-to-text and 708 MiB process GPU usage on RTX 5090, static batch 1. These are not edge-device measurements.

Historical token-selection checkpoints reached159/206 and later161/206 literal exact. The restart2 architecture has17,913,953 parameters, a2,048-patch input budget,512 selected patches after six blocks and two AR layers. These are different checkpoints/training histories, not one lossless model update. Standard R/RS options are unchanged; notebook9 exposes the separate experimental workflow. Reference weights are not bundled.

### Latest research terminal: restart2
Historical token-selection checkpoints reached159/206 and later161/206 literal exact. The restart2 architecture has17,913,953 parameters, a2,048-patch input budget,512 selected patches after six blocks and two AR layers. These are different checkpoints/training histories, not one lossless model update. Standard R/RS options are unchanged; notebook9 exposes the separate experimental workflow. Reference weights are not bundled.
The change is not lossless:16 strings differ, including six exact gains and four losses. Earlier strict0.05 arithmetic failures remain recorded; through-EOS output fidelity is separate from routing or calibration equivalence. Reference weights remain unbundled; no edge or rotation claim.

Literal exact requires every character, currency symbol, punctuation mark and space in the expected sequence.


### Separate experimental workflow is now available

Use [notebook 9](9_experimental_compact_workflow.ipynb) and [the workflow guide](../docs/compact_workflow.md) for multistage distillation, not the one-step Florence recipe. One-epoch defaults are software demos; teacher provenance is your responsibility, the charset is limited, and automatic resume is absent. The exact reference checkpoint's current standalone public runtime v3 measured **1.356083 ms RGB /1.620985 ms encoded, 708 MiB**, preserving all 206 native strings. These are not results for newly trained demo weights. Native resolution flexibility does not imply arbitrary TensorRT budgets, dynamic batching, edge or rotation qualification.
